# QUANTBIT IDX Historical Data Pull v2.0**Fix semua gap dari v1: Gold, USDIDR, adjClose terpisah, retry, validation, format conversion**---**Estimasi waktu:** 45-60 menit untuk 100+ ticker × 11 tahun (2015-2025).**Output:**- `data/years/{2015..2025}.json` (long format: `{ticker: [records]}`)- `data/years_wide/{2015..2025}.json` (wide format siap import ke project)- `data/benchmark/{ihsg,gold,usdidr,gold_idr_per_gram}_2015_2025.json`- `data/dividends/{TICKER}.json`, `data/fundamentals/{TICKER}.json` (annual)- `data/quarterly/{TICKER}.json` (quarterly, NEW)- `data/splits/{TICKER}.json` (stock split history, audit adjClose)- `data/manifest/manifest.json` (summary metadata)- `data/DaftarSaham.csv` (master + sector + market cap)

In [ ]:
!pip install yfinance pandas requests tqdm -qimport yfinance as yfyf.shared._YF_BASE_URL = "https://query1.finance.yahoo.com"print('yfinance version:', yf.__version__)

In [ ]:
import os, jsondirs = [    'data/years', 'data/years_wide', 'data/dividends', 'data/fundamentals',    'data/quarterly', 'data/splits', 'data/benchmark', 'data/logs',    'data/manifest', '.checkpoint']for d in dirs:    os.makedirs(d, exist_ok=True)print('Direktori siap (10 folder)')

In [ ]:
import timefrom functools import wrapsfrom datetime import datetimedef retry(max_attempts=3, backoff=2, exceptions=(Exception,)):    def decorator(fn):        @wraps(fn)        def wrapper(*args, **kwargs):            for attempt in range(1, max_attempts + 1):                try:                    return fn(*args, **kwargs)                except exceptions as e:                    if attempt == max_attempts:                        print(f'  {fn.__name__} failed after {max_attempts}x: {e}')                        return None                    wait = backoff ** attempt                    print(f'  {fn.__name__} attempt {attempt}/{max_attempts}, retry in {wait}s')                    time.sleep(wait)        return wrapper    return decoratordef checkpoint(key, data, label='data'):    path = f'.checkpoint/{label}_{key}.json'    with open(path, 'w') as f:        json.dump(data, f, default=str)def load_checkpoint(key, label='data'):    path = f'.checkpoint/{label}_{key}.json'    if os.path.exists(path):        with open(path) as f:            return json.load(f)    return Nonedef log(msg):    ts = datetime.now().strftime('%H:%M:%S')    print(f'[{ts}] {msg}')print('Retry + checkpoint utilities loaded')

In [ ]:
import pandas as pdIDX_UNIVERSE = [    'BBCA','BBRI','BMRI','TLKM','ASII','UNVR','ICBP','INDF','KLBF','ANTM',    'PTBA','ITMG','SMGR','UNTR','JSMR','INCO','PGAS','PTPP','EXCL','ISAT',    'MNCN','GGRM','HMSP','CPIN','JPFA','TBIG','TOWR','SRTG','WIKA','WSKT',    'BRPT','BREN','AMMN','MEDC','ESSA','TPIA','INPP','KEJU','ACES','MAPI',    'MIKA','HRTA','ERAA','DMAS','PWON','BSDE','CTRA','SMRA','MYOR','ULTJ',    'AALI','ADRO','AKRA','AMRT','ARTO','BBNI','BBTN','BFIN','BJBR','BJTM',    'BKSL','BNGA','BNII','BRMS','BTPS','CSMI','CUAN','DEWA','DSNG',    'ELSA','EMTK','ENRG','GOTO','HEAL','HRUM','INKP','INTP','LSIP','MBMA',    'MDKA','MTEL','NISP','PGEO','PTRO','SIDO','TAPG','TKIM','WIIM',    'AADI','ADMR','BDMN','BRIS','CLEO','INDY','MAPA','MIDI','NELY','RIMO',    'WOOD','TRIL','TRAM','MYRX','KREN','SUGI','NUSA','WIFI','BUKA','BUMI',    'FILM','CUAN']seen = set()IDX_UNIVERSE = [t for t in IDX_UNIVERSE if not (t in seen or seen.add(t))]YAHOO_TICKERS = [f"{t}.JK" for t in IDX_UNIVERSE]print(f'Universe: {len(IDX_UNIVERSE)} ticker unik')print(f'Sample: {IDX_UNIVERSE[:10]} ... {IDX_UNIVERSE[-5:]}')

In [ ]:
from tqdm import tqdm@retry(max_attempts=3, backoff=2)def fetch_ticker_info(ticker_yahoo):    t = yf.Ticker(ticker_yahoo)    info = t.info or {}    return {        'yahoo': ticker_yahoo,        'name': info.get('longName') or info.get('shortName', ''),        'sector': info.get('sector', ''),        'industry': info.get('industry', ''),        'marketCap': info.get('marketCap'),        'employees': info.get('fullTimeEmployees'),        'currency': info.get('currency', 'IDR'),        'exchange': info.get('exchange', ''),        'quoteType': info.get('quoteType', ''),        'firstTradeDate': info.get('firstTradeDateMilliseconds', 0) // 1000,    }ticker_info = {}log(f'Start fetch ticker info for {len(YAHOO_TICKERS)} ticker')for t in tqdm(YAHOO_TICKERS, desc='Info'):    info = fetch_ticker_info(t)    if info:        code = t.replace('.JK', '')        ticker_info[code] = info    time.sleep(0.3)with open('data/manifest/ticker_info.json', 'w') as f:    json.dump(ticker_info, f, default=str)log(f'{len(ticker_info)} ticker info saved')

In [ ]:
START_DATE = '2015-01-01'END_DATE   = '2025-12-31'YEARS      = list(range(2015, 2026))all_data = load_checkpoint('ohlcv_v2', 'data') or {str(y): {} for y in YEARS}@retry(max_attempts=4, backoff=2)def fetch_ohlcv(ticker_yahoo):    df = yf.download(        ticker_yahoo,        start=START_DATE, end=END_DATE,        auto_adjust=False,        progress=False, timeout=30    )    if df.empty:        raise ValueError('empty dataframe')    return dffailed_ohlcv = []log('Start OHLCV fetch (this takes ~15-20 min for 100+ tickers)')for ticker in tqdm(YAHOO_TICKERS, desc='OHLCV'):    code = ticker.replace('.JK', '')    if all(code in all_data[str(y)] for y in YEARS):        continue    try:        df = fetch_ohlcv(ticker)        if df is None or df.empty:            failed_ohlcv.append(ticker)            continue        if isinstance(df.columns, pd.MultiIndex):            df.columns = df.columns.get_level_values(0)        df.index = pd.to_datetime(df.index)        for year in YEARS:            year_df = df[df.index.year == year]            if year_df.empty:                continue            records = []            for date, row in year_df.iterrows():                records.append({                    'date':   date.strftime('%Y-%m-%d'),                    'open':   round(float(row['Open']),  2) if pd.notna(row['Open'])  else None,                    'high':   round(float(row['High']),  2) if pd.notna(row['High'])  else None,                    'low':    round(float(row['Low']),   2) if pd.notna(row['Low'])   else None,                    'close':  round(float(row['Close']), 2) if pd.notna(row['Close']) else None,                    'adjClose': round(float(row['Adj Close']), 2) if 'Adj Close' in row and pd.notna(row['Adj Close']) else None,                    'volume': int(row['Volume']) if pd.notna(row['Volume']) else 0                })            all_data[str(year)][code] = records        if len(ticker_info) % 10 == 0:            checkpoint('ohlcv_v2', all_data)    except Exception:        failed_ohlcv.append(ticker)    time.sleep(0.4)for year in YEARS:    with open(f'data/years/{year}.json', 'w') as f:        json.dump(all_data[str(year)], f)    print(f'  {year}.json: {len(all_data[str(year)])} tickers')log(f'Failed: {len(failed_ohlcv)} ticker: {failed_ohlcv[:5]}')

In [ ]:
@retry(max_attempts=3, backoff=2)def fetch_benchmark(symbol, name):    df = yf.download(symbol, start=START_DATE, end=END_DATE,                     auto_adjust=True, progress=False, timeout=30)    if df.empty:        raise ValueError(f'{name} empty')    if isinstance(df.columns, pd.MultiIndex):        df.columns = df.columns.get_level_values(0)    df.index = pd.to_datetime(df.index)    records = []    for date, row in df.iterrows():        records.append({            'date':   date.strftime('%Y-%m-%d'),            'open':   round(float(row['Open']),  2),            'high':   round(float(row['High']),  2),            'low':    round(float(row['Low']),   2),            'close':  round(float(row['Close']), 2),            'volume': int(row['Volume']) if pd.notna(row['Volume']) else 0        })    return recordslog('Fetching IHSG (^JKSE)')ihsg = fetch_benchmark('^JKSE', 'IHSG')with open('data/benchmark/ihsg_2015_2025.json', 'w') as f:    json.dump(ihsg, f)log(f'  IHSG: {len(ihsg)} days')log('Fetching Gold (GC=F)')gold = fetch_benchmark('GC=F', 'Gold')with open('data/benchmark/gold_2015_2025.json', 'w') as f:    json.dump(gold, f)log(f'  Gold: {len(gold)} days')log('Fetching USDIDR (USDIDR=X)')usdidr = fetch_benchmark('USDIDR=X', 'USDIDR')with open('data/benchmark/usdidr_2015_2025.json', 'w') as f:    json.dump(usdidr, f)log(f'  USDIDR: {len(usdidr)} days')log('Computing gold_IDR_per_gram ...')usdidr_map = {r['date']: r['close'] for r in usdidr}gold_idr = []for r in gold:    usd_idr = usdidr_map.get(r['date'])    if usd_idr:        idr_per_gram = (r['close'] * usd_idr) / 31.1035        gold_idr.append({            'date': r['date'],            'gold_usd_oz': r['close'],            'usdidr': usd_idr,            'gold_idr_gram': round(idr_per_gram, 0)        })with open('data/benchmark/gold_idr_per_gram_2015_2025.json', 'w') as f:    json.dump(gold_idr, f)log(f'  gold_IDR/gram: {len(gold_idr)} days')

In [ ]:
@retry(max_attempts=3, backoff=2)def fetch_dividends(ticker_yahoo):    t = yf.Ticker(ticker_yahoo)    divs = t.dividends    if divs is None or divs.empty:        return []    divs = divs[divs.index >= START_DATE]    return [{'date': d.strftime('%Y-%m-%d'), 'amount': round(float(v), 4)} for d, v in divs.items()]div_summary = {}for ticker in tqdm(YAHOO_TICKERS, desc='Dividends'):    code = ticker.replace('.JK', '')    out_path = f'data/dividends/{code}.json'    if os.path.exists(out_path):        with open(out_path) as f:            div_summary[code] = json.load(f)        continue    try:        records = fetch_dividends(ticker)        with open(out_path, 'w') as f:            json.dump(records, f)        div_summary[code] = records    except Exception:        div_summary[code] = []    time.sleep(0.3)has_div = sum(1 for v in div_summary.values() if v)log(f'{has_div}/{len(div_summary)} ticker punya dividen')

In [ ]:
@retry(max_attempts=3, backoff=2)def fetch_splits(ticker_yahoo):    t = yf.Ticker(ticker_yahoo)    splits = t.splits    if splits is None or splits.empty:        return []    return [{'date': d.strftime('%Y-%m-%d'), 'ratio': float(v)}            for d, v in splits.items() if d >= pd.Timestamp(START_DATE)]splits_summary = {}for ticker in tqdm(YAHOO_TICKERS, desc='Splits'):    code = ticker.replace('.JK', '')    try:        records = fetch_splits(ticker)        if records:            with open(f'data/splits/{code}.json', 'w') as f:                json.dump(records, f)        splits_summary[code] = records    except Exception:        splits_summary[code] = []    time.sleep(0.3)total_splits = sum(len(v) for v in splits_summary.values())log(f'{total_splits} stock split events')

In [ ]:
def safe_float(val):    try:        if pd.isna(val) or val is None:            return None        return float(val)    except Exception:        return Nonedef df_to_dict(df):    if df is None or df.empty:        return {}    result = {}    for col in df.columns:        year = str(col.year) if hasattr(col, 'year') else str(col)[:4]        result[year] = {str(metric): safe_float(df.loc[metric, col]) for metric in df.index}    return result@retry(max_attempts=3, backoff=2)def fetch_fundamentals_annual(ticker_yahoo):    t = yf.Ticker(ticker_yahoo)    return {        'income_stmt':   df_to_dict(t.financials),        'balance_sheet': df_to_dict(t.balance_sheet),        'cashflow':      df_to_dict(t.cashflow),    }@retry(max_attempts=3, backoff=2)def fetch_fundamentals_quarterly(ticker_yahoo):    t = yf.Ticker(ticker_yahoo)    return {        'income_stmt':   df_to_dict(t.quarterly_financials),        'balance_sheet': df_to_dict(t.quarterly_balance_sheet),        'cashflow':      df_to_dict(t.quarterly_cashflow),    }log('Fetching annual + quarterly fundamentals (~30-45 min)')for ticker in tqdm(YAHOO_TICKERS, desc='Fundamentals'):    code = ticker.replace('.JK', '')    annual_path = f'data/fundamentals/{code}.json'    if not os.path.exists(annual_path):        try:            ann = fetch_fundamentals_annual(ticker)            data = {'code': code, 'info': ticker_info.get(code, {}), **ann}            with open(annual_path, 'w') as f:                json.dump(data, f, default=str)        except Exception:            pass    quarterly_path = f'data/quarterly/{code}.json'    if not os.path.exists(quarterly_path):        try:            q = fetch_fundamentals_quarterly(ticker)            with open(quarterly_path, 'w') as f:                json.dump({**q, 'code': code}, f, default=str)        except Exception:            pass    time.sleep(0.5)

In [ ]:
import jsonlog('Running quality gate ...')issues = {'zero_volume': [], 'negative_price': [], 'date_gaps': [], 'missing_data': []}total_zero_vol = 0for year in YEARS:    with open(f'data/years/{year}.json') as f:        yd = json.load(f)    for ticker, recs in yd.items():        for r in recs:            if r.get('volume', 0) == 0:                issues['zero_volume'].append(f'{year}/{ticker}/{r["date"]}')                total_zero_vol += 1            for k in ('open','high','low','close','adjClose'):                v = r.get(k)                if v is not None and v < 0:                    issues['negative_price'].append(f'{year}/{ticker}/{r["date"]}/{k}={v}')    for ticker, recs in yd.items():        if not recs:            issues['missing_data'].append(f'{year}/{ticker} (empty)')            continue        if len(recs) < 100:            issues['date_gaps'].append(f'{year}/{ticker}: only {len(recs)} days')with open('data/benchmark/ihsg_2015_2025.json') as f:    ihsg = json.load(f)ihsg_dates = sorted([r['date'] for r in ihsg])log(f'  IHSG: {len(ihsg)} records, range {ihsg_dates[0]} -> {ihsg_dates[-1]}')for f_name in ('gold_2015_2025.json', 'usdidr_2015_2025.json', 'gold_idr_per_gram_2015_2025.json'):    with open(f'data/benchmark/{f_name}') as f:        d = json.load(f)    log(f'  {f_name}: {len(d)} records')log('QUALITY ISSUES:')log(f'  Zero-volume: {len(issues["zero_volume"])} ({total_zero_vol} total)')log(f'  Negative price: {len(issues["negative_price"])}')log(f'  Date gaps: {len(issues["date_gaps"])}')log(f'  Missing ticker: {len(issues["missing_data"])}')

In [ ]:
ihsg_map = {r['date']: r for r in ihsg}events = {    '2015-08-24': ('Yuan devaluation (low)', 4111, 'low'),    '2020-03-24': ('COVID crash (low)', 3912, 'low'),    '2018-12-24': ('Trade war bottom', 6015, 'low'),    '2015-01-02': ('IHSG 2015 open', 5242, 'close'),    '2025-12-30': ('IHSG 2025 close', 8647, 'close'),}log('Cross-check historical events:')all_ok = Truefor date, (label, expected, field) in events.items():    rec = ihsg_map.get(date)    if not rec:        print(f'  {date} missing')        all_ok = False        continue    actual = rec.get(field)    if abs(actual - expected) > 5:        print(f'  {date} {label}: expected {expected}, got {actual}')        all_ok = False    else:        print(f'  {date} {label}: {actual}')log('Result: ' + ('ALL EVENTS MATCH' if all_ok else 'SOME MISMATCHES'))

In [ ]:
manifest = {    'generated_at': datetime.now().isoformat(),    'ticker_universe': IDX_UNIVERSE,    'total_tickers': len(IDX_UNIVERSE),    'date_range': {'start': START_DATE, 'end': END_DATE},    'years': YEARS,    'data_files': {        'years_per_file': {},        'dividends_count': 0,        'fundamentals_count': 0,        'quarterly_count': 0,        'splits_total': 0,        'ticker_info_count': len(ticker_info),        'benchmarks': {}    },    'quality_summary': {        'zero_volume_total': total_zero_vol,        'negative_price_count': len(issues['negative_price']),        'date_gaps_count': len(issues['date_gaps']),        'missing_ticker_count': len(issues['missing_data'])    }}for y in YEARS:    with open(f'data/years/{y}.json') as f:        d = json.load(f)    n_tickers = len(d)    n_records = sum(len(v) for v in d.values())    manifest['data_files']['years_per_file'][y] = {        'tickers': n_tickers, 'records': n_records,        'size_kb': os.path.getsize(f'data/years/{y}.json') // 1024    }manifest['data_files']['dividends_count']  = len(os.listdir('data/dividends'))manifest['data_files']['fundamentals_count'] = len(os.listdir('data/fundamentals'))manifest['data_files']['quarterly_count']  = len(os.listdir('data/quarterly'))manifest['data_files']['splits_total'] = sum(    len(json.load(open(f'data/splits/{f}')))    for f in os.listdir('data/splits') if os.path.getsize(f'data/splits/{f}') > 2)for f_name in os.listdir('data/benchmark'):    with open(f'data/benchmark/{f_name}') as f:        d = json.load(f)    manifest['data_files']['benchmarks'][f_name] = len(d)with open('data/manifest/manifest.json', 'w') as f:    json.dump(manifest, f, indent=2)log('manifest.json saved')print(json.dumps(manifest['data_files'], indent=2))

In [ ]:
log('Converting to project format (year files -> wide format)...')with open('data/benchmark/ihsg_2015_2025.json') as f:    ihsg = json.load(f)with open('data/benchmark/gold_idr_per_gram_2015_2025.json') as f:    gold_idr = json.load(f)with open('data/benchmark/usdidr_2015_2025.json') as f:    usdidr = json.load(f)ihsg_map = {r['date']: r['close'] for r in ihsg}gold_map = {r['date']: r['gold_idr_gram'] for r in gold_idr}usdidr_map = {r['date']: r['close'] for r in usdidr}for year in YEARS:    with open(f'data/years/{year}.json') as f:        yd = json.load(f)    all_dates = sorted(set(r['date'] for recs in yd.values() for r in recs))    wide = []    for date in all_dates:        stockPrices = {}        stockAdjPrices = {}        stockVolumes = {}        stockOpens = {}        stockHighs = {}        stockLows = {}        for ticker, recs in yd.items():            rec = next((r for r in recs if r['date'] == date), None)            if rec:                stockPrices[ticker] = rec['close']                stockAdjPrices[ticker] = rec.get('adjClose') or rec['close']                stockVolumes[ticker] = rec['volume']                stockOpens[ticker] = rec['open']                stockHighs[ticker] = rec['high']                stockLows[ticker] = rec['low']        wide.append({            'date': date,            'ihsgPrice': ihsg_map.get(date),            'goldPrice': gold_map.get(date),            'usdidrRate': usdidr_map.get(date),            'stockPrices': stockPrices,            'stockAdjPrices': stockAdjPrices,            'stockVolumes': stockVolumes,            'stockOpens': stockOpens,            'stockHighs': stockHighs,            'stockLows': stockLows,        })    with open(f'data/years_wide/{year}.json', 'w') as f:        json.dump(wide, f)    log(f'  {year}: {len(wide)} days, {len(yd)} tickers')log('Done. Files di data/years_wide/ siap di-copy ke data/years/ project')

In [ ]:
import shutilfrom google.colab import filesmaster_df_full = pd.DataFrame([    {'Code': code, 'Yahoo': f'{code}.JK',     'Name': ticker_info.get(code, {}).get('name', ''),     'Sector': ticker_info.get(code, {}).get('sector', ''),     'MarketCap': ticker_info.get(code, {}).get('marketCap')}    for code in IDX_UNIVERSE])master_df_full.to_csv('data/DaftarSaham.csv', index=False)log(f'DaftarSaham.csv updated ({len(master_df_full)} rows)')backup_name = f'quantbit_data_2015_2025_{datetime.now().strftime("%Y%m%d_%H%M")}'shutil.copytree('data', backup_name, dirs_exist_ok=False)log(f'Backup: {backup_name}/')zip_name = 'quantbit_data_v2'shutil.make_archive(zip_name, 'zip', '.', 'data')size_mb = os.path.getsize(f'{zip_name}.zip') / 1024 / 1024log(f'{zip_name}.zip ({size_mb:.1f} MB)')files.download(f'{zip_name}.zip')log('Download dimulai...')

In [ ]:
print('=' * 60)print('   QUANTBIT DATA PULL v2.0 - FINAL REPORT')print('=' * 60)print()print(f'Range       : {START_DATE} -> {END_DATE}')print(f'Tickers     : {len(IDX_UNIVERSE)} ({manifest["data_files"]["ticker_info_count"]} with info)')print(f'OHLCV       : {sum(v["records"] for v in manifest["data_files"]["years_per_file"].values()):,} records')print(f'Dividends   : {manifest["data_files"]["dividends_count"]} files')print(f'Fundamentals: {manifest["data_files"]["fundamentals_count"]} annual + {manifest["data_files"]["quarterly_count"]} quarterly')print(f'Splits      : {manifest["data_files"]["splits_total"]} events')print()print('Benchmark:')for k, v in manifest['data_files']['benchmarks'].items():    print(f'  {k:50s}: {v:>5} days')print()print('Quality Gate:')for k, v in manifest['quality_summary'].items():    print(f'  {k:30s}: {v}')print()print('DONE. Data siap untuk ETL ke project QuantBit')print('Lanjut: copy data/years_wide/*.json ke project data/years/')print('Update DaftarSaham.csv di root project')